# Fine-Tuning Gemma2-V3-2B-it for Wordpress QA

## Install & Import Libraries

In [ ]:
!pip install -q -U peft evaluate transformers accelerate bitsandbytes peft
!pip install -U -q trl datasets wandb

In [ ]:
import torch
from transformers import (AutoTokenizer, 
                          AutoModelForCausalLM, 
                          BitsAndBytesConfig, 
                          AutoConfig,
                          TrainingArguments)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
from datasets import Dataset
import os
import pandas as pd
import wandb
from IPython.display import Markdown, display
import warnings

warnings.filterwarnings("ignore")

In [ ]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

if torch.cuda.is_available():
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    print("CUDA is available")
else:
    print("CUDA is not available")

In [ ]:
import wandb

wandb.init(mode="disabled")

## Preprocessing Datasets

### Load Datasets

In [ ]:
my_data = pd.read_json("/kaggle/input/wordpress-faq/wordpress_faq.json", orient="index")

print(my_data.head(3), sep="\n")

### Filter Based on Length

In [ ]:
question_column = "question"
answer_column = "answer"

my_data["text_length"] = my_data[question_column].str.len() + my_data[answer_column].str.len()

average_length = int(my_data["text_length"].mean())

shortest_length = int(my_data["text_length"].min())
longest_length = int(my_data["text_length"].max())

print("Average length of 'Question and Answer' in dataset:", average_length)
print("Shortest length of 'Question and Answer' in dataset:", shortest_length)
print("Longest length of 'Question and Answer' in dataset:", longest_length)

In [ ]:
median_text_length_threshold = int(my_data['text_length'].quantile(1))

filtered_data = my_data[my_data['text_length'] <= median_text_length_threshold]

print("Number of entries before filtering:", len(my_data))
print("Number of entries after filtering:", len(filtered_data))

print("---"*10)

max_text_length_in_filtered_data = int(filtered_data['text_length'].max())

print(f"Maximum text length before filtering: {longest_length}\nMaximum text length after filtering: {max_text_length_in_filtered_data}")

### Convert to Dataset object

In [ ]:
dataset = Dataset.from_pandas(filtered_data)
print(dataset)

## Preparing Model

### Load Model & Tokenizer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:",device)

In [ ]:
model_path = "/kaggle/input/gemma/transformers/2b-it/3"

bnbConfig = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path,
                                             device_map="auto",
                                             quantization_config=bnbConfig
                                            )

### LoRA Config

In [ ]:
lora_config = LoraConfig(
    r = 8,
    target_modules = ["q_proj", "o_proj", "k_proj", "v_proj",
                      "gate_proj", "up_proj", "down_proj"],
    task_type = "CAUSAL_LM",
)

### Prompt Formatting

In [ ]:
template = """Instruction:
{instruction}

Response:
{response}"""

def formatting_func(example):
    line = template.format(instruction=example[question_column], response=example[answer_column])

    return [line]

### Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    max_seq_length=512,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=50,
        learning_rate=2e-4,
        fp16=False,
        logging_steps=1,
        output_dir="outputs",
        optim="paged_adamw_8bit"
    ),
    peft_config=lora_config,
    formatting_func=formatting_func,
)

In [ ]:
trainer.train()

### Publish to HuggingFace

In [ ]:
os.environ["HF_TOKEN"] = "hugging face token key, you can create from your HF account."
model.push_to_hub("aminv/WordpressQA-Gemma2-v3-2B-it-FT", use_auth_token=os.getenv("HF_TOKEN"))
tokenizer.push_to_hub("aminv/WordpressQA-Gemma2-v3-2B-it-FT", use_auth_token=os.getenv("HF_TOKEN"))

### Save in Local

In [ ]:
model.save_pretrained("models/WordpressQA-Gemma2-v3-2B-it-FT")
tokenizer.save_pretrained("models/WordpressQA-Gemma2-v3-2B-it-FT")

## Testing Model

In [ ]:
template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

def generate_response(model, tokenizer, prompt, device, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to(device)

    outputs = model.generate(**inputs, num_return_sequences=1, max_new_tokens=max_new_tokens)

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    response_text = text.split("Response:")[1]
    
    return response_text

In [ ]:
question = "How i can install plugin in Wordpress?"

prompt = template.format(
    instruction=question,
    response="",
)

response_text = generate_response(trainer.model, tokenizer, prompt, device, 128)

Markdown(response_text)

# The End